In [ ]:
from data.DataLoader import DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

from models.LogisticRegression import LogisticRegressionPytorch

import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
dataloader = DataLoader()
X_train, X_test, y_train, y_test = dataloader.train_test_split(0.8, None)

## ScikitLearn Model

In [ ]:
onehot_hyperparams = {
    'categories': 'auto', 
    'drop': None, 
    'dtype': np.float64, 
    'handle_unknown': 'error', 
    'min_frequency': None, 
    'max_categories': None, 
    'feature_name_combiner': 'concat'
}

lr_hyperparams = {
    'penalty': 'l2',
    'dual': False,
    'tol': 1e-4,
    'C': 1,
    'fit_intercept': True,
    'intercept_scaling': 1,
    'class_weight': None,
    'random_state': None,
    'solver': 'lbfgs',
    'max_iter': 1000,
    'multi_class': 'auto',
    'verbose': 0,
    'warm_start': False,
    'n_jobs': None,
    'l1_ratio': None
}

In [ ]:
pipe = Pipeline(
    [
        ('stdscaler', StandardScaler()), 
        ('onehotencoder', OneHotEncoder(**onehot_hyperparams))
        ('lrmodel', LogisticRegression(**lr_hyperparams))
    ]
)

pipe.fit(X_train, y_train).score(X_test, y_test)

## Pytorch Model

In [ ]:
input_size = X_train.shape[1]
model = LogisticRegressionPytorch(input_size)

_hyperparams = {
    'lr': 0.01, 
    'momentum': 0,
    'dampening': 0,
    'weight_decay': 0,
    'nesterov': False
}

criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), **_hyperparams)

num_epochs = 1000

In [ ]:
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)

X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test)

In [ ]:
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor.view(-1, 1))
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

In [ ]:
model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor)
    predictions = (predictions > 0.5).float()

accuracy = (predictions == y_test_tensor.view(-1, 1)).sum().item() / len(y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

In [ ]:
# Export the model
torch.onnx.export(model,               # model being run
                  X_train_tensor,                         # model input (or a tuple for multiple inputs)
                  "model.onnx",   # where to save the model (can be a file or file-like object)
                  export_params=True,        # store the trained parameter weights inside the model file
                  #opset_version=10,          # the ONNX version to export the model to
                  #do_constant_folding=True,  # whether to execute constant folding for optimization
                  input_names = ['input'],   # the model's input names
                  output_names = ['output'], # the model's output names
                  dynamic_axes={'input' : {0 : 'batch_size'},    # variable length axes
                                'output' : {0 : 'batch_size'}})